In [7]:
# ===============================================
#   PIPELINE COMPLETO v37 – ENSEMBLE PERFEITO
# ===============================================

import pandas as pd
import numpy as np
import random
import warnings
import time
from datetime import datetime

!pip install catboost lightgbm xgboost --quiet

# MODELOS BASE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# STACKING
from sklearn.ensemble import StackingClassifier

# ML UTILS
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from joblib import dump

warnings.filterwarnings("ignore")

# =================================================
# 0. REPRODUTIBILIDADE
# =================================================
np.random.seed(42)
random.seed(42)

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

# =================================================
# 1. FUNÇÕES AUXILIARES
# =================================================

def formatar_celula(series_coluna):
    s = series_coluna.astype(str).replace("NULL", pd.NA)
    s = s.str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("utf-8")
    s = s.str.upper().str.replace(r"[^A-Z0-9]+", "_", regex=True).str.strip("_")
    s = s.replace("", pd.NA)
    return s

def adicionar_features_temporais(df):
    """Criação de lags e rolling statistics"""
    if "AVERAGE_FREE_FLOW_SPEED" in df.columns:
        df["SPEED_LAG1"] = df["AVERAGE_FREE_FLOW_SPEED"].shift(1)
        df["SPEED_LAG2"] = df["AVERAGE_FREE_FLOW_SPEED"].shift(2)
        df["SPEED_ROLL3"] = df["AVERAGE_FREE_FLOW_SPEED"].rolling(3).mean()
        df["SPEED_ROLL6"] = df["AVERAGE_FREE_FLOW_SPEED"].rolling(6).mean()
    return df

def preprocessar_dados(df, scaler=None):
    df = adicionar_features_temporais(df)

    if "AVERAGE_CLOUDINESS" in df.columns:
        df["AVERAGE_CLOUDINESS"] = formatar_celula(df["AVERAGE_CLOUDINESS"]).fillna("NONE")

    if "record_date" in df.columns:
        rd = pd.to_datetime(df["record_date"], dayfirst=True, errors="coerce")
        df["Hora_sin"] = np.sin(2*np.pi*rd.dt.hour/24)
        df["Hora_cos"] = np.cos(2*np.pi*rd.dt.hour/24)
        df["Mes_sin"] = np.sin(2*np.pi*rd.dt.month/12)
        df["Mes_cos"] = np.cos(2*np.pi*rd.dt.month/12)
        df["DIA_SEMANA"] = rd.dt.dayofweek
        df["IS_WEEKEND"] = (df["DIA_SEMANA"] >= 5).astype(int)

    df = df.drop(columns=["city_name","AVERAGE_RAIN","AVERAGE_PRECIPITATION","record_date"], errors="ignore")

    for col in ["LUMINOSITY","AVERAGE_CLOUDINESS","DIA_SEMANA"]:
        if col in df.columns:
            df = pd.get_dummies(df, columns=[col],
                                prefix=col[:3].upper() if col!="DIA_SEMANA" else "DAY",
                                dtype=int)

    cols_norm = [
        "AVERAGE_FREE_FLOW_SPEED","AVERAGE_TIME_DIFF","AVERAGE_FREE_FLOW_TIME",
        "AVERAGE_TEMPERATURE","AVERAGE_ATMOSP_PRESSURE","AVERAGE_HUMIDITY",
        "AVERAGE_WIND_SPEED","IS_WEEKEND","Hora_sin","Hora_cos",
        "Mes_sin","Mes_cos","SPEED_LAG1","SPEED_LAG2","SPEED_ROLL3","SPEED_ROLL6"
    ]

    cols_exist = [c for c in cols_norm if c in df.columns]

    if scaler is None:
        scaler = MinMaxScaler()
        if cols_exist:
            df[cols_exist] = scaler.fit_transform(df[cols_exist])
        return df, scaler
    else:
        if cols_exist:
            df[cols_exist] = scaler.transform(df[cols_exist])
        return df, scaler

# =================================================
# 2. LOAD TREINO
# =================================================
log("A carregar treino...")

df_train = pd.read_csv("datasets/training_data.csv", encoding="latin-1")
df_train["record_date"] = pd.to_datetime(df_train["record_date"], dayfirst=True)
df_train = df_train.sort_values("record_date").reset_index(drop=True)

# =================================================
# 3. TARGET COM ESCALA FIXA
# =================================================

# Escala final que pediste
fixed_classes = ["None","Low","Medium","High","Very_High"]

target_map = {
    "NONE":"None",
    "LOW":"Low",
    "MEDIUM":"Medium",
    "HIGH":"High",
    "VERY_HIGH":"Very_High"
}

y_raw = formatar_celula(df_train["AVERAGE_SPEED_DIFF"]).fillna("NONE")
y_fixed = y_raw.map(target_map).fillna("None")

le = LabelEncoder()
le.fit(fixed_classes)

y_train = le.transform(y_fixed)

# =================================================
# 4. PREPROCESS TREINO
# =================================================
log("A preprocessar treino...")
X_train, scaler_final = preprocessar_dados(df_train.drop(columns=["AVERAGE_SPEED_DIFF"]))
X_train = X_train.fillna(0)

# =================================================
# 5. MODELOS (COMBO PERFEITO)
# =================================================
log("A configurar ensemble perfeito...")

model_xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

model_lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=50,
    subsample=0.9,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model_cat = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.03,
    random_seed=42,
    verbose=False
)

meta_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=40,
    random_state=42,
    n_jobs=-1
)

stack_model = StackingClassifier(
    estimators=[("xgb", model_xgb), ("lgbm", model_lgbm), ("cat", model_cat)],
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1,
    stack_method="predict_proba",
    passthrough=True
)

# =================================================
# 6. CROSS-VALIDATION ACCURACY
# =================================================
log("A calcular accuracy cross-validation...")

tscv = TimeSeriesSplit(n_splits=5)

cv_scores = cross_val_score(stack_model, X_train, y_train, cv=tscv, scoring="accuracy", n_jobs=-1)

log(f"Scores por fold: {cv_scores}")
log(f"Accuracy média: {cv_scores.mean():.5f}")
log(f"Desvio padrão: {cv_scores.std():.5f}")

# =================================================
# 7. TREINO FINAL
# =================================================
log("Treino final...")
stack_model.fit(X_train, y_train)

# =================================================
# 8. TESTE + SUBMISSION
# =================================================
log("A carregar test_data...")

df_test = pd.read_csv("datasets/test_data.csv", encoding="latin-1")
df_test["record_date"] = pd.to_datetime(df_test["record_date"], dayfirst=True)

X_test, _ = preprocessar_dados(df_test, scaler=scaler_final)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

log("A gerar previsões...")

preds = stack_model.predict(X_test)
preds = le.inverse_transform(preds)

submission = pd.DataFrame({
    "RowId": np.arange(1, len(preds)+1),
    "Speed_Diff": preds
})

submission.to_csv("submission_v37.csv", index=False)
log("submission_v37.csv criado!")

# SAVE MODELS
dump(stack_model, "ensemble_v37.joblib")
dump(scaler_final, "scaler_v37.joblib")
dump(le, "label_encoder_v37.joblib")

log("=== FIM DO PIPELINE v37 ===")



[12:22:02] A carregar treino...
[12:22:02] A preprocessar treino...
[12:22:02] A configurar ensemble perfeito...
[12:22:02] A calcular accuracy cross-validation...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001250 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1407
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001689 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1599
[LightGBM] [Info] Number of data points in the train set: 1137, number of used features: 32
[LightGBM] [Info] Number of data points in the train set: 2272, number of used features: 32
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002836 seconds.
You can set `force_col_wise=true` to r